In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

#Load and preprocess the data:

data_path = "C:/Users/olufe/projects/Journal/dataset/HomeC_unsupervise_sim_combined_shuffled.csv"
data = pd.read_csv(data_path)

# Assuming the target column is named "Label", you can adjust the name accordingly
X = data.drop(columns=["Label"])
y = data["Label"]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [2]:
#Build the Deep Autoencoder:
def build_autoencoder(input_dim):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(64, activation='relu')(input_layer)
    encoded = Dense(32, activation='relu')(encoded)
    encoded = Dense(16, activation='relu')(encoded)
    decoded = Dense(32, activation='relu')(encoded)
    decoded = Dense(64, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    return autoencoder

# Assuming the input dimension is the number of features in the dataset
input_dim = X_train_scaled.shape[1]
autoencoder = build_autoencoder(input_dim)


In [3]:
#Train the autoencoder:
# Define early stopping to avoid overfitting
early_stopping = EarlyStopping(patience=3, restore_best_weights=True)

# Train the autoencoder
autoencoder.fit(X_train_scaled, X_train_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stopping])


Epoch 1/50
398/398 [==============================] - 3s 6ms/step - loss: 0.7774 - val_loss: 0.6315
Epoch 2/50
398/398 [==============================] - 2s 5ms/step - loss: 0.6184 - val_loss: 0.5606
Epoch 3/50
398/398 [==============================] - 2s 6ms/step - loss: 0.5651 - val_loss: 0.5254
Epoch 4/50
398/398 [==============================] - 2s 5ms/step - loss: 0.5364 - val_loss: 0.5071
Epoch 5/50
398/398 [==============================] - 2s 6ms/step - loss: 0.5176 - val_loss: 0.4917
Epoch 6/50
398/398 [==============================] - 2s 6ms/step - loss: 0.5046 - val_loss: 0.4806
Epoch 7/50
398/398 [==============================] - 2s 5ms/step - loss: 0.4951 - val_loss: 0.4757
Epoch 8/50
398/398 [==============================] - 2s 6ms/step - loss: 0.4884 - val_loss: 0.4678
Epoch 9/50
398/398 [==============================] - 2s 4ms/step - loss: 0.4829 - val_loss: 0.4629
Epoch 10/50
398/398 [==============================] - 2s 5ms/step - loss: 0.4779 - val_loss: 0.4623

In [4]:
#Evaluate the autoencoder:

# Obtain the reconstruction error on the test set
X_test_reconstructed = autoencoder.predict(X_test_scaled)
mse = np.mean(np.square(X_test_scaled - X_test_reconstructed), axis=1)

# Set a threshold to classify anomalies
threshold = np.percentile(mse, 95)
y_pred = [1 if err > threshold else 0 for err in mse]

# Evaluate performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
tnr = tn / (tn + fp)
fpr = fp / (tn + fp)
fnr = fn / (fn + tp)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, mse)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("TNR:", tnr)
print("FPR:", fpr)
print("FNR:", fnr)
print("F1-score:", f1)
print("AUC:", auc)

221/221 [==============================] - 1s 3ms/step
Accuracy: 0.9498867497168743
Precision: 0.03954802259887006
Recall: 0.5
TNR: 0.9516770892552586
FPR: 0.04832291074474133
FNR: 0.5
F1-score: 0.07329842931937174
AUC: 0.7991655161211727


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

#Load and preprocess the data:

data_path = "C:/Users/olufe/projects/Journal/dataset/HomeC_unsupervise_real_combined_shuffled.csv"
data = pd.read_csv(data_path)

# Assuming the target column is named "Label", you can adjust the name accordingly
X = data.drop(columns=["Label"])
y = data["Label"]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [6]:
#Build the Deep Autoencoder:
def build_autoencoder(input_dim):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(64, activation='relu')(input_layer)
    encoded = Dense(32, activation='relu')(encoded)
    encoded = Dense(16, activation='relu')(encoded)
    decoded = Dense(32, activation='relu')(encoded)
    decoded = Dense(64, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    return autoencoder

# Assuming the input dimension is the number of features in the dataset
input_dim = X_train_scaled.shape[1]
autoencoder = build_autoencoder(input_dim)

In [7]:
#Train the autoencoder:
# Define early stopping to avoid overfitting
early_stopping = EarlyStopping(patience=3, restore_best_weights=True)

# Train the autoencoder
autoencoder.fit(X_train_scaled, X_train_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stopping])

Epoch 1/50
592/592 [==============================] - 4s 5ms/step - loss: 0.6742 - val_loss: 0.5599
Epoch 2/50
592/592 [==============================] - 3s 5ms/step - loss: 0.5125 - val_loss: 0.4938
Epoch 3/50
592/592 [==============================] - 3s 4ms/step - loss: 0.4700 - val_loss: 0.4682
Epoch 4/50
592/592 [==============================] - 2s 4ms/step - loss: 0.4510 - val_loss: 0.4573
Epoch 5/50
592/592 [==============================] - 2s 4ms/step - loss: 0.4391 - val_loss: 0.4468
Epoch 6/50
592/592 [==============================] - 2s 3ms/step - loss: 0.4305 - val_loss: 0.4362
Epoch 7/50
592/592 [==============================] - 2s 4ms/step - loss: 0.4238 - val_loss: 0.4316
Epoch 8/50
592/592 [==============================] - 2s 4ms/step - loss: 0.4178 - val_loss: 0.4266
Epoch 9/50
592/592 [==============================] - 2s 3ms/step - loss: 0.4144 - val_loss: 0.4230
Epoch 10/50
592/592 [==============================] - 2s 4ms/step - loss: 0.4106 - val_loss: 0.4212

In [8]:
#Evaluate the autoencoder:

# Obtain the reconstruction error on the test set
X_test_reconstructed = autoencoder.predict(X_test_scaled)
mse = np.mean(np.square(X_test_scaled - X_test_reconstructed), axis=1)

# Set a threshold to classify anomalies
threshold = np.percentile(mse, 95)
y_pred = [1 if err > threshold else 0 for err in mse]

# Evaluate performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
tnr = tn / (tn + fp)
fpr = fp / (tn + fp)
fnr = fn / (fn + tp)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, mse)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("TNR:", tnr)
print("FPR:", fpr)
print("FNR:", fnr)
print("F1-score:", f1)
print("AUC:", auc)

329/329 [==============================] - 1s 3ms/step
Accuracy: 0.811834094368341
Precision: 0.2889733840304182
Recall: 0.08656036446469248
TNR: 0.957286432160804
FPR: 0.04271356783919598
FNR: 0.9134396355353075
F1-score: 0.13321647677475898
AUC: 0.6500654676363804
